# Exploration notebook

Template for cross-experiment analysis. Edit freely — gitignored after initial commit.
Promote any reusable pattern to `analysis/plot.py`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import analysis.plot as P

from analysis.load import load_results, filter_results, to_dataframe
from analysis.plot import metric_vs_param, convergence_curves, visualize

# ── Global switches ────────────────────────────────────────────
P.SAVE_FIGURES = False   # True  → save PNGs to OUTPUT_DIR
                         # False → display inline
STUDY_DIR  = '../results/aggregate'   # ← point here
OUTPUT_DIR = '../figs/explore'
RGB_INDICES = [20, 10, 5]            # bands for RGB preview
# ──────────────────────────────────────────────────────────────

PARAMS  = ['algorithm', 'lmbda', 'lmbda_m', 'p', 'q', 'r',
           'scale', 'noise_level', 'sigma_blur', 'max_iter', 'max_iter_cp']
METRICS = ['PSNR_mean', 'SSIM_mean', 'SAM_mean', 'RNMSE_mean', 'CC_mean']

results = load_results(STUDY_DIR)
df = to_dataframe(results)
print(f'{len(df)} experiments | {df["algorithm"].value_counts().to_dict()}')

## 1. Coverage — what has been run?

In [ ]:
# Full table of explored configurations
cols = [c for c in PARAMS if c in df.columns] + [c for c in METRICS if c in df.columns]
df[cols].sort_values('PSNR_mean', ascending=False)

In [ ]:
# Parameter space coverage — adjust x/y/hue/size to the axes you care about
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='lmbda', y='noise_level',
                hue='algorithm', size='PSNR_mean', sizes=(40, 200), ax=ax)
ax.set_xscale('log')
ax.set_title('Explored parameter space')
plt.tight_layout()

## 2. Define your slice

Set `FILTERS` once here. All cells below (`subset`, `subset_df`) use it.
Comment out keys to leave that dimension free.

In [ ]:
# ── Edit here ─────────────────────────────────────────────────
FILTERS = dict(
    algorithm   = 'CTV',
    noise_level = 40,
    # max_iter_cp = 50,
    # sigma_blur  = 1.0,
)
# ──────────────────────────────────────────────────────────────

subset    = filter_results(results, **FILTERS)
subset_df = to_dataframe(subset)

# Show what still varies in this slice (i.e. what you haven't fixed)
free = subset_df[[c for c in PARAMS if c in subset_df.columns]]
free_params = free.columns[free.nunique() > 1].tolist()
print(f'{len(subset)} experiments match')
print(f'Free parameters: {free_params}')
subset_df[[c for c in free_params + [m for m in METRICS if m in subset_df.columns]]]

## 3. Visual inspection

Set `i` to navigate through experiments in the current slice.
Re-run the cell to move to the next one.

In [ ]:
i = 0   # ← 0 … len(subset)-1

r = subset[i]
label = {k: r.get(k) for k in free_params + ['algorithm']}
print(f'Experiment {i}/{len(subset)-1}: {label}')
for m in METRICS:
    if m in r: print(f'  {m}: {r[m]:.4f}')

# Inputs + reconstruction (loads dataset from path stored in zarr attrs)
visualize(r['zarr_path'], output_dir=OUTPUT_DIR, rgb_indices=RGB_INDICES)

## 4. Metrics

In [ ]:
# Distributions for the current slice
metric_cols = [c for c in METRICS if c in subset_df.columns]
n = len(metric_cols)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
for ax, m in zip(axes, metric_cols):
    subset_df[m].plot.hist(ax=ax, bins=10, edgecolor='white')
    ax.set_title(m.replace('_mean', ''))
plt.suptitle(str(FILTERS), fontsize=9)
plt.tight_layout()

In [ ]:
# Metric vs one free parameter — change param and metric_name as needed
metric_vs_param(subset, param='lmbda', metric_name='PSNR', output_dir=OUTPUT_DIR)

## 5. Convergence

In [ ]:
# Lines by one free param — adjust group_by to a key from free_params
convergence_curves(subset, group_by='lmbda', facet_by='algorithm',
                   mode='distance', output_dir=OUTPUT_DIR)

## 6. Deep dive

Free-form. Apply additional `filter_results` calls, compare two slices, etc.

In [ ]:
# Example: compare two algorithms on the same fixed conditions
common = {k: v for k, v in FILTERS.items() if k != 'algorithm'}
ctv = filter_results(results, algorithm='CTV',        **common)
ga  = filter_results(results, algorithm='GradAlign',  **common)
if ctv and ga:
    from analysis.plot import compare_algorithms
    compare_algorithms({'CTV': ctv, 'GradAlign': ga}, 'PSNR', output_dir=OUTPUT_DIR)

In [ ]:
# Two-parameter interaction heatmap (works well when grid is regular)
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(subset_df['lmbda'], subset_df['noise_level'],
                c=subset_df['PSNR_mean'], cmap='viridis', s=80)
plt.colorbar(sc, ax=ax, label='PSNR')
ax.set_xscale('log')
ax.set_xlabel('lmbda'); ax.set_ylabel('noise_level')
ax.set_title('PSNR over (lmbda, noise_level) — current slice')
plt.tight_layout()